# Cross-Lingual Sentiment Analysis 

Multilingual Amazon reviews sentiment classifier using XLM-RoBERTa. This notebook loads English, French, and Spanish subsets of the Amazon Reviews Multi dataset, performs preprocessing and tokenization, fine-tunes an XLM-RoBERTa sequence classification model for 5-class sentiment prediction, evaluates monolingual performance on English, and conducts zero-shot transfer evaluation on French and Spanish with basic error analysis and result tables.

**References**

- Keung, P., Lu, Y., Szarvas, G. (2020). “The Multilingual Amazon Reviews Corpus.” In Proceedings of EMNLP 2020.  
  [https://huggingface.co/datasets/amazon_reviews_multi](https://huggingface.co/datasets/amazon_reviews_multi)

- Conneau, A., Khandelwal, K., Goyal, N., et al. (2020). “Unsupervised Cross-lingual Representation Learning at Scale.” In Proceedings of ACL 2020 (XLM-RoBERTa).  
  [https://huggingface.co/xlm-roberta-base](https://huggingface.co/xlm-roberta-base)


# Data Preprocessing

Load datasets

In [ ]:
from datasets import load_dataset

dataset_en = load_dataset("mteb/amazon_reviews_multi", "en")
dataset_fr = load_dataset("mteb/amazon_reviews_multi", "fr")
dataset_es = load_dataset("mteb/amazon_reviews_multi", "es")


Split datasets

In [2]:
dataset_en_train = dataset_en['train']
dataset_en_val   = dataset_en['validation']
dataset_en_test  = dataset_en['test']

dataset_fr_train = dataset_fr['train']
dataset_fr_val   = dataset_fr['validation']
dataset_fr_test  = dataset_fr['test']

dataset_es_train = dataset_es['train']
dataset_es_val   = dataset_es['validation']
dataset_es_test  = dataset_es['test']

# For example, take 20k samples from each train set
dataset_en_train_small = dataset_en_train.shuffle(seed=42).select(range(20000))
dataset_fr_train_small = dataset_fr_train.shuffle(seed=42).select(range(20000))
dataset_es_train_small = dataset_es_train.shuffle(seed=42).select(range(20000))

# Keep the original validation and test sets
dataset_en_val_small = dataset_en_val
dataset_en_test_small = dataset_en_test

dataset_fr_val_small = dataset_fr_val
dataset_fr_test_small = dataset_fr_test

dataset_es_val_small = dataset_es_val
dataset_es_test_small = dataset_es_test

In [3]:
print("English samples:", len(dataset_en_train_small))
print("French samples:", len(dataset_fr_train_small))
print("Spanish samples:", len(dataset_es_train_small))

English samples: 20000
French samples: 20000
Spanish samples: 20000


In [4]:
print(dataset_en_train_small[0])

{'id': 'en_0976163', 'text': 'Worked in front position, not rear\n\n3 stars because these are not rear brakes as stated in the item description. At least the mount adapter only worked on the front fork of the bike that I got it for.', 'label': 2, 'label_text': '2'}


In [5]:
# The label is already 0-4 (corresponding to 1-5 stars)
# 0 -> 1 star, 1 -> 2 stars, ..., 4 -> 5 stars
# So we can train directly for 5-class classification

# Example from the reduced English training set
print("Example English review:", dataset_en_train_small[0]['text'])
print("Label (0-4):", dataset_en_train_small[0]['label'])

Example English review: Worked in front position, not rear

3 stars because these are not rear brakes as stated in the item description. At least the mount adapter only worked on the front fork of the bike that I got it for.
Label (0-4): 2


#   Monolingual

 Tokenize Text

In [38]:
from transformers import XLMRobertaTokenizer

# Load tokenizer
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

# Tokenization function
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=128)

# Helper function: tokenize + return torch-formatted COPY
def prepare_for_trainer(dataset):
    tokenized = dataset.map(tokenize, batched=True)
    tokenized = tokenized.with_format(
        type='torch',
        columns=['input_ids', 'attention_mask', 'label']
    )
    return tokenized


# Create TORCH datasets (copies)


# English
dataset_en_train_torch = prepare_for_trainer(dataset_en_train_small)
dataset_en_val_torch   = prepare_for_trainer(dataset_en_val_small)
dataset_en_test_torch  = prepare_for_trainer(dataset_en_test_small)

# French
dataset_fr_train_torch = prepare_for_trainer(dataset_fr_train_small)
dataset_fr_val_torch   = prepare_for_trainer(dataset_fr_val_small)
dataset_fr_test_torch  = prepare_for_trainer(dataset_fr_test_small)

# Spanish
dataset_es_train_torch = prepare_for_trainer(dataset_es_train_small)
dataset_es_val_torch   = prepare_for_trainer(dataset_es_val_small)
dataset_es_test_torch  = prepare_for_trainer(dataset_es_test_small)

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Model Setup

In [7]:
from transformers import XLMRobertaForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = XLMRobertaForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=5  # 5-class classification
).to(device)

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Metrics

In [44]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {"accuracy": acc, "f1": f1}

Training & Evaluation

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True  # if using GPU (recommended)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_en_train_torch,   
    eval_dataset=dataset_en_test_torch,     
    compute_metrics=compute_metrics
)

trainer.train()

  0%|          | 0/3750 [00:00<?, ?it/s]

{'loss': 1.569, 'grad_norm': 18.990264892578125, 'learning_rate': 1.9466666666666668e-05, 'epoch': 0.08}
{'loss': 1.2114, 'grad_norm': 9.600171089172363, 'learning_rate': 1.8933333333333334e-05, 'epoch': 0.16}
{'loss': 1.0667, 'grad_norm': 14.183547973632812, 'learning_rate': 1.8400000000000003e-05, 'epoch': 0.24}
{'loss': 1.0338, 'grad_norm': 14.350021362304688, 'learning_rate': 1.7866666666666666e-05, 'epoch': 0.32}
{'loss': 1.0204, 'grad_norm': 23.012920379638672, 'learning_rate': 1.7333333333333336e-05, 'epoch': 0.4}
{'loss': 0.96, 'grad_norm': 24.247825622558594, 'learning_rate': 1.6800000000000002e-05, 'epoch': 0.48}
{'loss': 0.961, 'grad_norm': 9.220996856689453, 'learning_rate': 1.6266666666666668e-05, 'epoch': 0.56}
{'loss': 0.9554, 'grad_norm': 12.061484336853027, 'learning_rate': 1.5733333333333334e-05, 'epoch': 0.64}
{'loss': 0.9282, 'grad_norm': 16.581573486328125, 'learning_rate': 1.5200000000000002e-05, 'epoch': 0.72}
{'loss': 0.9606, 'grad_norm': 12.662498474121094, 'le

  0%|          | 0/157 [00:00<?, ?it/s]

{'eval_loss': 0.9049881100654602, 'eval_accuracy': 0.6292, 'eval_f1': 0.6237946649595617, 'eval_runtime': 36.1802, 'eval_samples_per_second': 138.197, 'eval_steps_per_second': 4.339, 'epoch': 1.0}
{'loss': 0.8556, 'grad_norm': 14.173542022705078, 'learning_rate': 1.3066666666666668e-05, 'epoch': 1.04}
{'loss': 0.8163, 'grad_norm': 12.075155258178711, 'learning_rate': 1.2533333333333336e-05, 'epoch': 1.12}
{'loss': 0.7981, 'grad_norm': 12.507460594177246, 'learning_rate': 1.2e-05, 'epoch': 1.2}
{'loss': 0.8259, 'grad_norm': 16.524816513061523, 'learning_rate': 1.1466666666666668e-05, 'epoch': 1.28}
{'loss': 0.7901, 'grad_norm': 17.730457305908203, 'learning_rate': 1.0933333333333334e-05, 'epoch': 1.36}
{'loss': 0.8383, 'grad_norm': 16.877182006835938, 'learning_rate': 1.04e-05, 'epoch': 1.44}
{'loss': 0.7907, 'grad_norm': 13.093188285827637, 'learning_rate': 9.866666666666668e-06, 'epoch': 1.52}
{'loss': 0.781, 'grad_norm': 9.013468742370605, 'learning_rate': 9.333333333333334e-06, 'epo

  0%|          | 0/157 [00:00<?, ?it/s]

{'eval_loss': 0.8588384985923767, 'eval_accuracy': 0.6396, 'eval_f1': 0.637203475221743, 'eval_runtime': 35.4412, 'eval_samples_per_second': 141.079, 'eval_steps_per_second': 4.43, 'epoch': 2.0}
{'loss': 0.6671, 'grad_norm': 16.192665100097656, 'learning_rate': 6.133333333333334e-06, 'epoch': 2.08}
{'loss': 0.7271, 'grad_norm': 65.47994232177734, 'learning_rate': 5.600000000000001e-06, 'epoch': 2.16}
{'loss': 0.6488, 'grad_norm': 8.776960372924805, 'learning_rate': 5.0666666666666676e-06, 'epoch': 2.24}
{'loss': 0.707, 'grad_norm': 14.005197525024414, 'learning_rate': 4.533333333333334e-06, 'epoch': 2.32}
{'loss': 0.6927, 'grad_norm': 18.697959899902344, 'learning_rate': 4.000000000000001e-06, 'epoch': 2.4}
{'loss': 0.7109, 'grad_norm': 32.23216247558594, 'learning_rate': 3.4666666666666672e-06, 'epoch': 2.48}
{'loss': 0.6634, 'grad_norm': 25.05524253845215, 'learning_rate': 2.9333333333333338e-06, 'epoch': 2.56}
{'loss': 0.6558, 'grad_norm': 28.605422973632812, 'learning_rate': 2.4000

  0%|          | 0/157 [00:00<?, ?it/s]

{'eval_loss': 0.9049744009971619, 'eval_accuracy': 0.6404, 'eval_f1': 0.6386335308396601, 'eval_runtime': 36.406, 'eval_samples_per_second': 137.34, 'eval_steps_per_second': 4.312, 'epoch': 3.0}
{'train_runtime': 4629.2023, 'train_samples_per_second': 12.961, 'train_steps_per_second': 0.81, 'train_loss': 0.8398861307779948, 'epoch': 3.0}


TrainOutput(global_step=3750, training_loss=0.8398861307779948, metrics={'train_runtime': 4629.2023, 'train_samples_per_second': 12.961, 'train_steps_per_second': 0.81, 'total_flos': 3946772136960000.0, 'train_loss': 0.8398861307779948, 'epoch': 3.0})

In [10]:
# Directory where you want to save
save_dir = "./xlmr_sentiment_en"

# Save model
model.save_pretrained(save_dir)

# Save tokenizer
tokenizer.save_pretrained(save_dir)

print(f"Model and tokenizer saved to {save_dir}")

Model and tokenizer saved to ./xlmr_sentiment_en


Evaluate on English test set

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
from transformers import Trainer, AutoModelForSequenceClassification

# Load saved model
model_en = AutoModelForSequenceClassification.from_pretrained("./xlmr_sentiment_en")

# Define compute_metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')  # weighted F1 for multi-class
    return {"accuracy": acc, "f1": f1}


trainer_en = Trainer(model=model_en, compute_metrics=compute_metrics)

# Evaluate on English test set
results_en = trainer_en.evaluate(eval_dataset=dataset_en_test_torch)


df_results_en = pd.DataFrame({
    "Metric": ["Loss", "Accuracy (%)", "F1 (%)"],
    "Value": [
        round(results_en['eval_loss'], 4),
        round(results_en['eval_accuracy'] * 100, 2),
        round(results_en['eval_f1'] * 100, 2)
    ]
})

print(df_results_en)

  0%|          | 0/625 [00:00<?, ?it/s]

         Metric   Value
0          Loss   0.905
1  Accuracy (%)  64.040
2        F1 (%)  63.860


Zero-shot evaluation on French and Spanish

In [48]:
import pandas as pd

# French
results_fr = trainer_en.evaluate(eval_dataset=dataset_fr_test_torch)
# Spanish
results_es = trainer_en.evaluate(eval_dataset=dataset_es_test_torch)

# Gather results
results_dict = {
    "English": results_en,
    "French (zero-shot)": results_fr,
    "Spanish (zero-shot)": results_es
}

# Create table
metrics = ["Loss", "Accuracy (%)", "F1 (%)"]
table_data = {"Metric": metrics}

for lang, res in results_dict.items():
    table_data[lang] = [
        round(res['eval_loss'], 4),
        round(res['eval_accuracy'] * 100, 2),
        round(res['eval_f1'] * 100, 2)
    ]

df_comparison = pd.DataFrame(table_data)
print(df_comparison)


  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

         Metric  English  French (zero-shot)  Spanish (zero-shot)
0          Loss    0.905              1.1337               1.1259
1  Accuracy (%)   64.040             55.3200              55.0200
2        F1 (%)   63.860             55.1900              54.9500


Error analysis

In [ ]:
from torch import argmax
import torch
import pandas as pd

# Make predictions using trainer
#preds = trainer.predict(dataset_en_test_torch).predictions
pred_labels = argmax(torch.tensor(preds), dim=1)

# True labels
true_labels = dataset_en_test_torch['label']

# Raw texts
raw_texts = dataset_en_test_small['text']  

# Collect misclassified examples
misclassified = []
for i in range(len(true_labels)):
    if pred_labels[i] != true_labels[i]:
        misclassified.append({
            'text': raw_texts[i],
            'true_label': true_labels[i].item(),
            'pred_label': pred_labels[i].item()
        })

# Show first 10 misclassified examples
for ex in misclassified[:10]:
    print("Text:", ex['text'])
    print("True label:", ex['true_label'], "| Predicted:", ex['pred_label'])
    print("-"*50)

# convert to DataFrame for easier inspection

df_misclassified = pd.DataFrame(misclassified)
df_misclassified.head(10)

Text: Gold filled earrings

You want an HONEST answer? I just returned from UPS where I returned the FARCE of an earring set to Amazon. It did NOT look like what I saw on Amazon. Only a baby would be able to wear the size of the earring. They were SO small. the size of a pin head I at first thought Amazon had forgotten to enclose them in the bag! I didn't bother to take them out of the bag and you can have them back. Will NEVER order another thing from your company. A disgrace. Honest enough for you? Grandma
True label: 0 | Predicted: 1
--------------------------------------------------
Text: Poor container

The glue works fine but the container is impossible to work with. The cap doesn't come off without plyers and then won't go back on without a violent abrupt force involving both hands and a solid object (desk drawer). This happened even though I was careful to not gum up the lid or tapering snout.
True label: 0 | Predicted: 1
--------------------------------------------------
Text:

,text,true_label,pred_label
0,Gold filled earrings\n\nYou want an HONEST ans...,0,1
1,Poor container\n\nThe glue works fine but the ...,0,1
2,Horrible\n\nDoesn’t even work . Did nothing fo...,0,1
3,One Star\n\nNever received item. Given a refund.,0,2
4,DONT BUY!!!\n\nTerrible!!!! I couldnt even att...,0,1
5,Package arrived in empty.\n\nMy package arrive...,0,1
6,Belt loops\n\nBelt loops were not fastened upo...,0,1
7,One Star\n\nThe battery covers screw stripped ...,0,1
8,One Star\n\nIt didn’t work at all. All the wax...,0,1
9,Sunflower flavor is horrible\n\nGuess i picked...,0,1


Count misclassifications by true → predicted

In [ ]:
from collections import Counter

# True → Predicted pairs
tp_pairs = [(true_labels[i].item(), pred_labels[i].item()) 
            for i in range(len(true_labels)) if pred_labels[i] != true_labels[i]]

Counter(tp_pairs)

Counter({(3, 4): 386,
         (1, 2): 329,
         (2, 3): 291,
         (0, 1): 260,
         (4, 3): 211,
         (1, 0): 164,
         (2, 1): 157,
         (3, 2): 115,
         (0, 2): 92,
         (1, 3): 54,
         (2, 4): 49,
         (2, 0): 41,
         (4, 2): 21,
         (3, 1): 15,
         (0, 3): 12,
         (0, 4): 10,
         (1, 4): 10,
         (3, 0): 8,
         (4, 0): 6,
         (4, 1): 3})

Group misclassified texts by predicted label

In [54]:
import pandas as pd

df_misclassified = pd.DataFrame(misclassified)
df_misclassified.groupby('pred_label')['text'].count()

pred_label
0    219
1    435
2    557
3    568
4    455
Name: text, dtype: int64

Inspect top misclassified examples per class

In [55]:
for pred in df_misclassified['pred_label'].unique():
    print(f"\n=== Predicted as {pred} ===\n")
    for t in df_misclassified[df_misclassified['pred_label']==pred]['text'][:5]:
        print(t)
        print("-"*50)


=== Predicted as 1 ===

Gold filled earrings

You want an HONEST answer? I just returned from UPS where I returned the FARCE of an earring set to Amazon. It did NOT look like what I saw on Amazon. Only a baby would be able to wear the size of the earring. They were SO small. the size of a pin head I at first thought Amazon had forgotten to enclose them in the bag! I didn't bother to take them out of the bag and you can have them back. Will NEVER order another thing from your company. A disgrace. Honest enough for you? Grandma
--------------------------------------------------
Poor container

The glue works fine but the container is impossible to work with. The cap doesn't come off without plyers and then won't go back on without a violent abrupt force involving both hands and a solid object (desk drawer). This happened even though I was careful to not gum up the lid or tapering snout.
--------------------------------------------------
Horrible

Doesn’t even work . Did nothing for me :

# Joint multilingual fine-tuning

Combine datasets

In [2]:
from datasets import concatenate_datasets

# Combine training sets
train_multi = concatenate_datasets([
    dataset_en_train_small,
    dataset_fr_train_small,
    dataset_es_train_small
])

# Combine validation sets
val_multi = concatenate_datasets([
    dataset_en_val_small,
    dataset_fr_val_small,
    dataset_es_val_small
])

print("Combined multilingual training samples:", len(train_multi))
print("Combined multilingual validation samples:", len(val_multi))

Combined multilingual training samples: 60000
Combined multilingual validation samples: 15000


In [18]:
print(dataset_en_train_small[0])

{'id': 'en_0976163', 'text': 'Worked in front position, not rear\n\n3 stars because these are not rear brakes as stated in the item description. At least the mount adapter only worked on the front fork of the bike that I got it for.', 'label': 2, 'label_text': '2'}


Tokenizer

In [4]:
from transformers import XLMRobertaTokenizer

tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=128)

# Apply tokenizer to the combined multilingual dataset
train_multi = train_multi.map(tokenize, batched=True)
val_multi = val_multi.map(tokenize, batched=True)

# Format for PyTorch
train_multi.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_multi.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Load XLM-R for sequence classification

In [5]:
from transformers import XLMRobertaForSequenceClassification

# 5 labels for 5-star sentiment
model_multi = XLMRobertaForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=5
)

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Define metrics

In [6]:
from sklearn.metrics import accuracy_score, f1_score
import torch

def compute_metrics(p):
    preds = torch.tensor(p.predictions).argmax(dim=1)
    labels = torch.tensor(p.label_ids)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {"accuracy": acc, "f1": f1}

Training arguments

In [7]:
from transformers import TrainingArguments

training_args_multi = TrainingArguments(
    output_dir="./results_multi",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs_multi",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

Trainer

In [8]:
from transformers import Trainer

trainer_multi = Trainer(
    model=model_multi,
    args=training_args_multi,
    train_dataset=train_multi,
    eval_dataset=val_multi,
    compute_metrics=compute_metrics
)

Train the model

In [9]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("No GPU detected, using CPU")

GPU detected: NVIDIA GeForce RTX 3050 Laptop GPU


In [10]:
trainer_multi.train()

  0%|          | 0/11250 [00:00<?, ?it/s]

{'loss': 1.6141, 'grad_norm': 18.152976989746094, 'learning_rate': 1.9822222222222226e-05, 'epoch': 0.03}
{'loss': 1.2563, 'grad_norm': 7.196776390075684, 'learning_rate': 1.9644444444444447e-05, 'epoch': 0.05}
{'loss': 1.1744, 'grad_norm': 64.33045959472656, 'learning_rate': 1.9466666666666668e-05, 'epoch': 0.08}
{'loss': 1.1004, 'grad_norm': 21.058622360229492, 'learning_rate': 1.928888888888889e-05, 'epoch': 0.11}
{'loss': 1.0913, 'grad_norm': 23.86844253540039, 'learning_rate': 1.9111111111111113e-05, 'epoch': 0.13}
{'loss': 1.083, 'grad_norm': 9.985457420349121, 'learning_rate': 1.8933333333333334e-05, 'epoch': 0.16}
{'loss': 1.0277, 'grad_norm': 17.588197708129883, 'learning_rate': 1.8755555555555558e-05, 'epoch': 0.19}
{'loss': 1.0411, 'grad_norm': 17.940481185913086, 'learning_rate': 1.857777777777778e-05, 'epoch': 0.21}
{'loss': 1.0541, 'grad_norm': 17.889753341674805, 'learning_rate': 1.8400000000000003e-05, 'epoch': 0.24}
{'loss': 0.9951, 'grad_norm': 30.78723907470703, 'lea

  0%|          | 0/469 [00:00<?, ?it/s]

{'eval_loss': 0.931961178779602, 'eval_accuracy': 0.5903333333333334, 'eval_f1': 0.5935468117680139, 'eval_runtime': 107.9253, 'eval_samples_per_second': 138.985, 'eval_steps_per_second': 4.346, 'epoch': 1.0}
{'loss': 0.9087, 'grad_norm': 11.363055229187012, 'learning_rate': 1.3244444444444447e-05, 'epoch': 1.01}
{'loss': 0.8711, 'grad_norm': 11.902262687683105, 'learning_rate': 1.3066666666666668e-05, 'epoch': 1.04}
{'loss': 0.8687, 'grad_norm': 15.154829978942871, 'learning_rate': 1.288888888888889e-05, 'epoch': 1.07}
{'loss': 0.8759, 'grad_norm': 10.085468292236328, 'learning_rate': 1.2711111111111112e-05, 'epoch': 1.09}
{'loss': 0.8418, 'grad_norm': 15.706501960754395, 'learning_rate': 1.2533333333333336e-05, 'epoch': 1.12}
{'loss': 0.8843, 'grad_norm': 12.47640323638916, 'learning_rate': 1.2355555555555557e-05, 'epoch': 1.15}
{'loss': 0.8535, 'grad_norm': 10.906576156616211, 'learning_rate': 1.217777777777778e-05, 'epoch': 1.17}
{'loss': 0.9007, 'grad_norm': 9.723434448242188, 'le

  0%|          | 0/469 [00:00<?, ?it/s]

{'eval_loss': 0.9128429889678955, 'eval_accuracy': 0.6063333333333333, 'eval_f1': 0.6004369695382435, 'eval_runtime': 106.977, 'eval_samples_per_second': 140.217, 'eval_steps_per_second': 4.384, 'epoch': 2.0}
{'loss': 0.7783, 'grad_norm': 12.680706024169922, 'learning_rate': 6.488888888888889e-06, 'epoch': 2.03}
{'loss': 0.7836, 'grad_norm': 18.35831069946289, 'learning_rate': 6.311111111111111e-06, 'epoch': 2.05}
{'loss': 0.7683, 'grad_norm': 20.7933292388916, 'learning_rate': 6.133333333333334e-06, 'epoch': 2.08}
{'loss': 0.764, 'grad_norm': 12.649826049804688, 'learning_rate': 5.955555555555555e-06, 'epoch': 2.11}
{'loss': 0.7936, 'grad_norm': 13.612607955932617, 'learning_rate': 5.777777777777778e-06, 'epoch': 2.13}
{'loss': 0.755, 'grad_norm': 14.029011726379395, 'learning_rate': 5.600000000000001e-06, 'epoch': 2.16}
{'loss': 0.7768, 'grad_norm': 13.335429191589355, 'learning_rate': 5.422222222222223e-06, 'epoch': 2.19}
{'loss': 0.7673, 'grad_norm': 18.632286071777344, 'learning_r

  0%|          | 0/469 [00:00<?, ?it/s]

{'eval_loss': 0.9350674748420715, 'eval_accuracy': 0.6073333333333333, 'eval_f1': 0.6047834305601045, 'eval_runtime': 107.8143, 'eval_samples_per_second': 139.128, 'eval_steps_per_second': 4.35, 'epoch': 3.0}
{'train_runtime': 13818.9066, 'train_samples_per_second': 13.026, 'train_steps_per_second': 0.814, 'train_loss': 0.8758370225694444, 'epoch': 3.0}


TrainOutput(global_step=11250, training_loss=0.8758370225694444, metrics={'train_runtime': 13818.9066, 'train_samples_per_second': 13.026, 'train_steps_per_second': 0.814, 'total_flos': 1.184031641088e+16, 'train_loss': 0.8758370225694444, 'epoch': 3.0})

Evaluate multilingual model

In [16]:
dataset_en_test_small = dataset_en_test_small.map(tokenize, batched=True)

dataset_en_test_small.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'label']
)

dataset_fr_test_small = dataset_fr_test_small.map(tokenize, batched=True)

dataset_fr_test_small.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'label']
)



dataset_es_test_small = dataset_es_test_small.map(tokenize, batched=True)

dataset_es_test_small.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'label']
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
# English
results_en_multi = trainer_multi.evaluate(eval_dataset=dataset_en_test_small)
# French
results_fr_multi = trainer_multi.evaluate(eval_dataset=dataset_fr_test_small)
# Spanish
results_es_multi = trainer_multi.evaluate(eval_dataset=dataset_es_test_small)


data = {
    "Language": ["English", "French", "Spanish"],
    "Eval Loss": [
        results_en_multi['eval_loss'],
        results_fr_multi['eval_loss'],
        results_es_multi['eval_loss']
    ],
    "Eval Accuracy (%)": [
        results_en_multi['eval_accuracy'] * 100,
        results_fr_multi['eval_accuracy'] * 100,
        results_es_multi['eval_accuracy'] * 100
    ],
    "Eval F1 (%)": [
        results_en_multi['eval_f1'] * 100,
        results_fr_multi['eval_f1'] * 100,
        results_es_multi['eval_f1'] * 100
    ]
}


df_results = pd.DataFrame(data)
df_results = df_results.round(2)

print(df_results)


  0%|          | 0/157 [00:00<?, ?it/s]

  0%|          | 0/157 [00:00<?, ?it/s]

  0%|          | 0/157 [00:00<?, ?it/s]

  Language  Eval Loss  Eval Accuracy (%)  Eval F1 (%)
0  English       0.87              64.92        64.71
1   French       0.95              59.12        58.82
2  Spanish       0.94              60.62        60.29


Save multilingual model

In [12]:
save_dir_multi = "./xlmr_sentiment_multi"
model_multi.save_pretrained(save_dir_multi)
print(f"Multilingual model saved to {save_dir_multi}")

Multilingual model saved to ./xlmr_sentiment_multi
